In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("D:\car-price-predictor\data\cleaned_cars.csv")

X = df.drop(columns=["selling_price"])
y = np.log1p(df["selling_price"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
print()
print(X_train.columns.tolist())

<>:5: SyntaxWarning: invalid escape sequence '\c'
<>:5: SyntaxWarning: invalid escape sequence '\c'
C:\Users\sai charan\AppData\Local\Temp\ipykernel_40984\4015636910.py:5: SyntaxWarning: invalid escape sequence '\c'
  df = pd.read_csv("D:\car-price-predictor\data\cleaned_cars.csv")


X_train: (5368, 11)
X_test: (1343, 11)
y_train: (5368,)
y_test: (1343,)

['year', 'km_driven', 'fuel', 'seller_type', 'transmission', 'owner', 'mileage', 'engine', 'max_power', 'seats', 'brand']


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

num_cols = ["year", "km_driven", "mileage", "engine", "max_power", "seats"]
cat_cols = ["brand", "fuel", "seller_type", "transmission", "owner"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Before:", X_train.shape)
print("After:", X_train_processed.shape)
print("Test after:", X_test_processed.shape)

Before: (5368, 11)
After: (5368, 50)
Test after: (1343, 50)


In [3]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(model, X_tr, y_tr, X_te, y_te):
    pred_tr = model.predict(X_tr)
    pred_te = model.predict(X_te)
    return {
        "Train R2 (log)": r2_score(y_tr, pred_tr),
        "Test R2 (log)": r2_score(y_te, pred_te),
        "Test MAE (Rs)": mean_absolute_error(np.expm1(y_te), np.expm1(pred_te)),
        "Test RMSE (Rs)": np.sqrt(mean_squared_error(np.expm1(y_te), np.expm1(pred_te))),
    }

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

lin_preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

lin_model = Pipeline([
    ("preprocess", lin_preprocessor),
    ("model", LinearRegression()),
])

lin_model.fit(X_train, y_train)

results = evaluate(lin_model, X_train, y_train, X_test, y_test)
for name, value in results.items():
    print(f"{name}: {value:,.3f}")

Train R2 (log): 0.893
Test R2 (log): 0.878
Test MAE (Rs): 96,837.139
Test RMSE (Rs): 187,759.695


In [4]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

dt_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", DecisionTreeRegressor(random_state=42)),
])

rf_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
])

for name, m in [("Decision Tree", dt_model), ("Random Forest", rf_model)]:
    m.fit(X_train, y_train)
    print(name)
    for k, v in evaluate(m, X_train, y_train, X_test, y_test).items():
        print(f"  {k}: {v:,.3f}")
    print()

Decision Tree
  Train R2 (log): 0.998
  Test R2 (log): 0.841
  Test MAE (Rs): 94,470.800
  Test RMSE (Rs): 171,517.052

Random Forest
  Train R2 (log): 0.987
  Test R2 (log): 0.907
  Test MAE (Rs): 74,515.379
  Test RMSE (Rs): 138,048.736



In [5]:
from sklearn.model_selection import cross_val_score

for name, m in [("Linear Regression", lin_model), ("Random Forest", rf_model)]:
    scores = cross_val_score(m, X_train, y_train, cv=5, scoring="r2")
    print(name)
    print("  Fold scores:", np.round(scores, 3))
    print(f"  Mean: {scores.mean():.3f}   Std: {scores.std():.3f}")
    print()

Linear Regression
  Fold scores: [0.896 0.884 0.889 0.888 0.891]
  Mean: 0.890   Std: 0.004

Random Forest
  Fold scores: [0.921 0.907 0.916 0.91  0.912]
  Mean: 0.913   Std: 0.005



In [6]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "model__max_depth": [10, 15, 20, None],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": [0.3, 0.5, 0.7, 1.0],
}

search = RandomizedSearchCV(
    rf_model,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring="r2",
    random_state=42,
    n_jobs=1,
)
search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print(f"Best CV R2: {search.best_score_:.3f}")

best_rf = search.best_estimator_
for k, v in evaluate(best_rf, X_train, y_train, X_test, y_test).items():
    print(f"  {k}: {v:,.3f}")

Best params: {'model__min_samples_leaf': 2, 'model__max_features': 0.5, 'model__max_depth': None}
Best CV R2: 0.916
  Train R2 (log): 0.970
  Test R2 (log): 0.914
  Test MAE (Rs): 73,300.486
  Test RMSE (Rs): 135,600.939


In [7]:
import joblib, json, os

os.makedirs("../models", exist_ok=True)
joblib.dump(rf_model, "../models/car_price_model.joblib", compress=3)

options = {col: sorted(df[col].dropna().unique().tolist()) for col in cat_cols}
with open("../models/form_options.json", "w") as f:
    json.dump(options, f, indent=2)

# Test: reload the model and predict one test car
loaded = joblib.load("../models/car_price_model.joblib")
sample = X_test.iloc[[0]]
print(sample.to_string())
print("Predicted price:", round(float(np.expm1(loaded.predict(sample))[0])))
print("Actual price:   ", round(float(np.expm1(y_test.iloc[0]))))
print("Model file size (MB):", round(os.path.getsize("../models/car_price_model.joblib") / 1e6, 1))

      year  km_driven    fuel seller_type transmission        owner  mileage  engine  max_power  seats   brand
4995  2017    40000.0  Diesel  Individual       Manual  First Owner    23.59  1364.0      67.05    5.0  Toyota
Predicted price: 617334
Actual price:    550000
Model file size (MB): 10.3
